# 03 – Waterbirds: ERM Baseline (ResNet-18)

This notebook trains a ResNet-18 model with standard Empirical Risk Minimization (ERM) on the Waterbirds dataset. The trained model weights are saved and used as the backbone in notebook 04 for DFR retraining.


In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
import numpy as np
from sklearn.metrics import accuracy_score
from wilds import get_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.manual_seed(0); np.random.seed(0)


Device: cpu


In [2]:
# 1) Load Waterbirds Dataset

dataset = get_dataset(dataset="waterbirds", download=False)

img_size = 224

# Training: random crops + horizontal flip for data augmentation
train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
# Evaluation: center crop only, no augmentation
eval_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = dataset.get_subset("train", transform=train_tfms)
val_data   = dataset.get_subset("val",   transform=eval_tfms)
test_data  = dataset.get_subset("test",  transform=eval_tfms)

# metadata_fields includes ['y', 'background']; we need the background column index
print("metadata_fields:", dataset.metadata_fields)
place_idx = dataset.metadata_fields.index("background")  # 0=land, 1=water

batch_size = 64
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)


metadata_fields: ['background', 'y', 'from_source_domain']


In [3]:
## 2) Model: ResNet-18 (pretrained on ImageNet) + 2-class head
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
in_feats = resnet.fc.in_features
resnet.fc = nn.Linear(in_feats, 2)  # Replace final layer: 2 classes (landbird vs waterbird)
model = resnet.to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()


In [4]:
## 3) Evaluation Helper: Mean and Worst-Group Accuracy
# Groups are defined by (label y, background): g = 2*y + background ∈ {0,1,2,3}
@torch.no_grad()
def evaluate_mean_worst(model, loader, place_idx, tag="set"):
    model.eval()
    all_y, all_pred, all_g = [], [], []
    for x, y, meta in loader:
        x = x.to(device)
        logits = model(x)
        pred = logits.argmax(1).cpu().numpy()
        y = y.numpy()
        place = meta[:, place_idx].numpy().astype(int)  # 0=land, 1=water
        g = (y.astype(int) * 2 + place).astype(int)    # group index

        all_y.append(y); all_pred.append(pred); all_g.append(g)

    y = np.concatenate(all_y)
    p = np.concatenate(all_pred)
    g = np.concatenate(all_g)

    mean = (p == y).mean()
    accs = {gi: (p[g == gi] == y[g == gi]).mean() for gi in np.unique(g)}
    worst = min(accs.values())
    print(f"[{tag}] mean={mean:.3f}, worst={worst:.3f}, groups={accs}")
    return mean, worst, accs


In [5]:
# 4) ERM Training Loop
# Train for 5 epochs as a quick baseline. Increase to 10-20 for better performance.
epochs = 5
for ep in range(1, epochs + 1):
    model.train()
    running = 0.0
    for x, y, meta in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()
        running += loss.item() * x.size(0)
    print(f"Epoch {ep} | train_loss={running / len(train_loader.dataset):.4f}")

    # Evaluate mean and worst-group accuracy on the validation set
    evaluate_mean_worst(model, val_loader, place_idx, tag=f"Val @ep{ep}")


Epoch 1 | train_loss=0.1663
[Val @ep1] mean=0.824, worst=0.647, groups={0: 0.9678800856531049, 1: 0.6952789699570815, 2: 0.6466165413533834, 3: 0.9473684210526315}
Epoch 2 | train_loss=0.0640
[Val @ep2] mean=0.810, worst=0.631, groups={0: 0.974304068522484, 1: 0.630901287553648, 2: 0.6917293233082706, 3: 0.9774436090225563}
Epoch 3 | train_loss=0.0391
[Val @ep3] mean=0.841, worst=0.421, groups={0: 0.9935760171306209, 1: 0.7854077253218884, 2: 0.42105263157894735, 3: 0.9172932330827067}
Epoch 4 | train_loss=0.0252
[Val @ep4] mean=0.872, worst=0.571, groups={0: 0.9807280513918629, 1: 0.8347639484978541, 2: 0.5714285714285714, 3: 0.924812030075188}
Epoch 5 | train_loss=0.0229
[Val @ep5] mean=0.780, worst=0.556, groups={0: 0.9764453961456103, 1: 0.5944206008583691, 2: 0.556390977443609, 3: 0.9624060150375939}


In [6]:
# 5) Final Test Evaluation (ERM)
print("=== ERM (ResNet-18) — Final Test Results ===")
evaluate_mean_worst(model, test_loader, place_idx, tag="Test (ERM)")


=== ERM (ResNet18) — Final Test ===
[Test (ERM)] mean=0.818, worst=0.639, groups={0: 0.9866962305986696, 1: 0.6580931263858093, 2: 0.6386292834890965, 3: 0.9626168224299065}


(0.8175698998964446,
 0.6386292834890965,
 {0: 0.9866962305986696,
  1: 0.6580931263858093,
  2: 0.6386292834890965,
  3: 0.9626168224299065})

In [7]:
# 6) Save ERM Weights (for DFR retraining in notebook 04)
torch.save(model.state_dict(), "waterbirds_erm_resnet18.pth")
print("Saved: waterbirds_erm_resnet18.pth")


Kaydedildi: waterbirds_erm_resnet18.pth
